<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/CNN_minst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

In [14]:
torch.manual_seed(42)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [16]:
import kagglehub
import pandas as pd
import os

# Download latest version of a dataset known to contain fer2013.csv
df_path = kagglehub.dataset_download("nicolejyt/facialexpressionrecognition")

# Search for fer2013.csv within the downloaded directory structure
csv_file_found = False
for root, dirs, files in os.walk(df_path):
    if "fer2013.csv" in files:
        csv_full_path = os.path.join(root, "fer2013.csv")
        df = pd.read_csv(csv_full_path)
        csv_file_found = True
        print(f"Found fer2013.csv at: {csv_full_path}")
        print("DataFrame loaded successfully. Shape:", df.shape)
        break

if not csv_file_found:
    print(f"Error: fer2013.csv not found in {df_path} or its subdirectories.")
    print("Contents of the downloaded path:", os.listdir(df_path))

print("Path to dataset files root:", df_path)

Using Colab cache for faster access to the 'facialexpressionrecognition' dataset.
Found fer2013.csv at: /kaggle/input/facialexpressionrecognition/fer2013.csv
DataFrame loaded successfully. Shape: (35887, 3)
Path to dataset files root: /kaggle/input/facialexpressionrecognition


In [17]:
X = df['pixels'].apply(lambda x: np.array(x.split(), dtype='float32'))
y = df.iloc[:, 0].values

X_train, X_test, y_train, y_test = train_test_split(np.stack(X.values), y, test_size=0.2, random_state=42)

X_train = X_train/255.0
X_test = X_test/255.0

In [22]:
class CustomDataset(Dataset):
  def __init__(self,features, labels):
      self.features = torch.tensor(features, dtype = torch.float32).reshape(-1,1,48,48)
      self.labels = torch.tensor(labels, dtype=torch.long)


  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    return self.features[idx], self.labels[idx]

In [29]:
#create test_dataset object
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)


In [24]:
train_loader = DataLoader(train_dataset,batch_size =32, shuffle = True)
test_loader = DataLoader(test_dataset,batch_size =32, shuffle = True)

In [31]:
from torch.nn.modules.pooling import MaxPool2d
from torch.nn.modules.activation import ReLU
class MyNN(nn.Module):
  def __init__(self,input_features):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(input_features,32,kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32,64,kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2)

    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*12*12,128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10)
    )


  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)

    return x

In [32]:
epochs = 100
learning_rate = 0.01

In [33]:
model = MyNN(1)
model = model.to(device)

#loss function
criterion = nn.CrossEntropyLoss()

#optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [34]:


for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.699305043719659
Epoch: 2 , Loss: 1.4882064186916053
Epoch: 3 , Loss: 1.3922319438781399
Epoch: 4 , Loss: 1.3107997696091709
Epoch: 5 , Loss: 1.2305127676982923
Epoch: 6 , Loss: 1.16212956331886
Epoch: 7 , Loss: 1.082933728041787
Epoch: 8 , Loss: 0.9996126527775635
Epoch: 9 , Loss: 0.9330600773571329
Epoch: 10 , Loss: 0.8536922756109577
Epoch: 11 , Loss: 0.7819510761872431
Epoch: 12 , Loss: 0.7120356953356473
Epoch: 13 , Loss: 0.6388515920641693
Epoch: 14 , Loss: 0.578773795197695
Epoch: 15 , Loss: 0.5368039888767463
Epoch: 16 , Loss: 0.48162829897517884
Epoch: 17 , Loss: 0.43991799406159693
Epoch: 18 , Loss: 0.40639206791972793
Epoch: 19 , Loss: 0.3800734585039069
Epoch: 20 , Loss: 0.3502204253572664
Epoch: 21 , Loss: 0.3364831872433756
Epoch: 22 , Loss: 0.3171215713156889
Epoch: 23 , Loss: 0.2920399620560701
Epoch: 24 , Loss: 0.2765563349768453
Epoch: 25 , Loss: 0.25642449477426127
Epoch: 26 , Loss: 0.24483151972824851
Epoch: 27 , Loss: 0.22411179489856442
Epoch: 28

In [35]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=9216, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [36]:

#evaluation code
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    _, predicted = torch.max(outputs.data, 1)
    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the test set: {accuracy:.2f}%')

Accuracy on the test set: 55.71%


In [37]:

#evaluation code
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    outputs = model(batch_features)
    _, predicted = torch.max(outputs.data, 1)
    total += batch_labels.size(0)
    correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the test set: {accuracy:.2f}%')

Accuracy on the test set: 99.79%
